# Notebook 11 — Full Analysis Pipeline

This capstone notebook reproduces the full `main.py` pipeline with commentary, then demonstrates how to design and compare custom policy scenarios.

Sections:
1. Base case analysis — the Italian system as-is
2. Technology sweeps — nuclear, solar, coal
3. Fuel price sensitivity
4. Custom scenario comparison — three policy paths
5. Synthesis — what does it all mean?

**Runtime**: ~2-3 minutes (reduced MC runs)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

from energy_sim.config import (
    ITALIAN_MIX, GAS_SCENARIOS, COAL_SCENARIOS, CO2_SCENARIOS,
    P_PEAK_GW, INTERCONNECTIONS, PRICE_AREAS,
    PRICE_AREA_CORRELATIONS, STORAGE_UNITS,
)
from energy_sim.simulation import (
    run_monte_carlo, sweep_technology,
)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

N_RUNS = 10  # Reduced for notebook interactivity (use 100 for publication)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

## 1. Base Case: Italy today

Run the full system with all features enabled: interconnections + storage.

In [ ]:
mc_base = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=N_RUNS, seed=42,
    interconnections_cfg=INTERCONNECTIONS,
    price_areas_cfg=PRICE_AREAS,
    price_area_correlations=PRICE_AREA_CORRELATIONS,
    storage_cfg=STORAGE_UNITS,
)

print("=== BASE CASE RESULTS ===")
print(f"  Electricity price:         {mc_base['avg_price'].mean():.1f} +/- {mc_base['avg_price'].std():.1f} EUR/MWh")
print(f"  Carbon intensity (terr):   {mc_base['carbon_intensity'].mean():.0f} gCO2/kWh")
print(f"  Carbon intensity (cons):   {mc_base['carbon_intensity_consumption'].mean():.0f} gCO2/kWh")
print(f"  Total emissions:           {mc_base['total_emissions'].mean()/1e6:.2f} Mt CO2")
print(f"  System inertia:            {mc_base['avg_inertia'].mean():.2f} s")
print(f"  Storage revenue:           {mc_base['storage_revenue_eur'][:,0].mean()/1e6:+.1f} M EUR/year")

# Net imports
net_twh = mc_base['net_import_twh'].mean(axis=0)
for name, twh in zip(mc_base['interconnection_names'], net_twh):
    direction = 'import' if twh > 0 else 'export'
    print(f"  {name}: {twh:+.2f} TWh ({direction})")

In [ ]:
# Monthly price pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean_monthly = mc_base['monthly_prices'].mean(axis=0)
std_monthly = mc_base['monthly_prices'].std(axis=0)
axes[0].bar(month_names, mean_monthly, yerr=std_monthly,
            color='steelblue', alpha=0.7, capsize=3)
axes[0].set_ylabel('EUR/MWh')
axes[0].set_title('Monthly electricity prices (base case)')

# Price histogram
axes[1].hist(mc_base['avg_price'], bins=8, color='steelblue',
             alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Annual average price (EUR/MWh)')
axes[1].set_ylabel('MC runs')
axes[1].set_title('Price distribution across MC runs')

plt.tight_layout()
plt.show()

## 2. Technology Sweep: Nuclear vs Solar

Compare the effect of adding nuclear vs adding solar capacity:

In [ ]:
pcts = np.array([0, 5, 10, 15, 20])

nuc_sweep = sweep_technology(ITALIAN_MIX, 'nuclear', pcts,
                              GAS_SCENARIOS['base'], n_runs=N_RUNS, seed=42)
sol_sweep = sweep_technology(ITALIAN_MIX, 'solar', pcts,
                              GAS_SCENARIOS['base'], n_runs=N_RUNS, seed=42)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for sweep, label, color in [(nuc_sweep, 'Nuclear', 'purple'), (sol_sweep, 'Solar', 'gold')]:
    p = [r['pct'] for r in sweep]
    
    axes[0].errorbar(p, [r['mean_price'] for r in sweep],
                     yerr=[r['std_price'] for r in sweep],
                     fmt='o-', color=color, capsize=4, lw=2, label=label)
    axes[1].plot(p, [r['mean_carbon_intensity'] for r in sweep],
                'o-', color=color, lw=2, label=label)
    axes[2].plot(p, [r['mean_inertia'] for r in sweep],
                'o-', color=color, lw=2, label=label)

axes[0].set_xlabel('Penetration (%)')
axes[0].set_ylabel('EUR/MWh')
axes[0].set_title('Electricity price')
axes[0].legend()

axes[1].set_xlabel('Penetration (%)')
axes[1].set_ylabel('gCO2/kWh')
axes[1].set_title('Carbon intensity')
axes[1].legend()

axes[2].set_xlabel('Penetration (%)')
axes[2].set_ylabel('H (s)')
axes[2].set_title('System inertia')
axes[2].axhline(3.5, color='red', ls='--', alpha=0.5, label='H_min')
axes[2].legend()

plt.suptitle('Nuclear vs Solar: head-to-head comparison', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Custom Scenario Comparison

Let's compare three concrete policy paths for Italy:

1. **Status quo** — current mix, no changes
2. **Nuclear path** — add 20% nuclear, keep interconnections + storage
3. **Renewables + storage path** — add 30% solar + 10% wind, double storage capacity

In [ ]:
total_cap = sum(v['capacity_gw'] for v in ITALIAN_MIX.values())

# Scenario 1: Status quo
mix_sq = deepcopy(ITALIAN_MIX)

# Scenario 2: Nuclear path
mix_nuc = deepcopy(ITALIAN_MIX)
mix_nuc['nuclear']['capacity_gw'] = total_cap * 0.20

# Scenario 3: RES + storage
mix_res = deepcopy(ITALIAN_MIX)
mix_res['solar']['capacity_gw'] += total_cap * 0.30
mix_res['wind']['capacity_gw'] += total_cap * 0.10

stor_double = deepcopy(STORAGE_UNITS)
for k in stor_double:
    stor_double[k]['energy_capacity_gwh'] *= 2
    stor_double[k]['power_capacity_gw'] *= 2

scenarios = {
    'Status quo': (mix_sq, STORAGE_UNITS),
    'Nuclear (20%)': (mix_nuc, STORAGE_UNITS),
    'RES+Storage (x2)': (mix_res, stor_double),
}

results = {}
for name, (mix, stor) in scenarios.items():
    print(f"\n--- Running: {name} ---")
    mc = run_monte_carlo(
        mix, GAS_SCENARIOS['base'],
        n_runs=N_RUNS, seed=42,
        interconnections_cfg=INTERCONNECTIONS,
        price_areas_cfg=PRICE_AREAS,
        price_area_correlations=PRICE_AREA_CORRELATIONS,
        storage_cfg=stor,
    )
    results[name] = mc

In [ ]:
# Summary table
print(f"\n{'Scenario':<25s} {'Price':>8s} {'CI terr':>8s} {'CI cons':>8s} "
      f"{'Emissions':>10s} {'Inertia':>8s}")
print(f"{'':25s} {'EUR/MWh':>8s} {'gCO2':>8s} {'gCO2':>8s} {'Mt':>10s} {'s':>8s}")
print("-" * 75)
for name, mc in results.items():
    print(f"{name:<25s} "
          f"{mc['avg_price'].mean():8.1f} "
          f"{mc['carbon_intensity'].mean():8.0f} "
          f"{mc['carbon_intensity_consumption'].mean():8.0f} "
          f"{mc['total_emissions'].mean()/1e6:10.2f} "
          f"{mc['avg_inertia'].mean():8.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scenario_names = list(results.keys())
colors = ['steelblue', 'purple', 'gold']

# Price comparison
for i, (name, mc) in enumerate(results.items()):
    axes[0].bar(i, mc['avg_price'].mean(), color=colors[i],
                yerr=mc['avg_price'].std(), capsize=5, alpha=0.7)
axes[0].set_xticks(range(len(scenario_names)))
axes[0].set_xticklabels(scenario_names, rotation=15, ha='right')
axes[0].set_ylabel('EUR/MWh')
axes[0].set_title('Electricity price')

# Carbon intensity
for i, (name, mc) in enumerate(results.items()):
    axes[1].bar(i, mc['carbon_intensity'].mean(), color=colors[i], alpha=0.7)
axes[1].set_xticks(range(len(scenario_names)))
axes[1].set_xticklabels(scenario_names, rotation=15, ha='right')
axes[1].set_ylabel('gCO2/kWh')
axes[1].set_title('Carbon intensity')

# Monthly price comparison
for i, (name, mc) in enumerate(results.items()):
    axes[2].plot(range(12), mc['monthly_prices'].mean(axis=0),
                'o-', color=colors[i], lw=2, label=name)
axes[2].set_xticks(range(12))
axes[2].set_xticklabels(month_names, fontsize=8)
axes[2].set_ylabel('EUR/MWh')
axes[2].set_title('Monthly prices by scenario')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Stress test: gas crisis

How do our three scenarios perform under a gas price crisis?

In [ ]:
results_crisis = {}
for name, (mix, stor) in scenarios.items():
    mc = run_monte_carlo(
        mix, GAS_SCENARIOS['crisis'],  # mu=90 EUR/MWh_th
        n_runs=N_RUNS, seed=42,
        interconnections_cfg=INTERCONNECTIONS,
        price_areas_cfg=PRICE_AREAS,
        price_area_correlations=PRICE_AREA_CORRELATIONS,
        storage_cfg=stor,
    )
    results_crisis[name] = mc

print(f"\n{'Scenario':<25s} {'Base price':>10s} {'Crisis price':>12s} {'Delta':>8s}")
print("-" * 60)
for name in scenario_names:
    p_base = results[name]['avg_price'].mean()
    p_crisis = results_crisis[name]['avg_price'].mean()
    print(f"{name:<25s} {p_base:10.1f} {p_crisis:12.1f} {p_crisis-p_base:+8.1f}")

## 5. Design your own scenario

Modify the parameters below to create and test your own policy scenario:

In [ ]:
# ── DESIGN YOUR SCENARIO ─────────────────────────────────
my_mix = deepcopy(ITALIAN_MIX)
my_mix['nuclear']['capacity_gw'] = 10.0     # GW nuclear
my_mix['solar']['capacity_gw'] = 50.0       # GW solar
my_mix['wind']['capacity_gw'] = 20.0        # GW wind
my_mix['gas']['capacity_gw'] = 35.0         # GW gas (reduced)
my_mix['coal']['capacity_gw'] = 0.0         # GW coal

my_gas_scenario = GAS_SCENARIOS['base']      # or 'tension', 'crisis'
my_storage = STORAGE_UNITS                   # or None to disable
# ─────────────────────────────────────────────────────────

my_mc = run_monte_carlo(
    my_mix, my_gas_scenario,
    n_runs=N_RUNS, seed=42,
    interconnections_cfg=INTERCONNECTIONS,
    price_areas_cfg=PRICE_AREAS,
    price_area_correlations=PRICE_AREA_CORRELATIONS,
    storage_cfg=my_storage,
)

print(f"=== YOUR SCENARIO ===")
print(f"  Mix: {', '.join(f'{k}={v["capacity_gw"]:.0f}GW' for k, v in my_mix.items() if v['capacity_gw'] > 0)}")
print(f"  Price:     {my_mc['avg_price'].mean():.1f} +/- {my_mc['avg_price'].std():.1f} EUR/MWh")
print(f"  CI:        {my_mc['carbon_intensity'].mean():.0f} gCO2/kWh")
print(f"  Emissions: {my_mc['total_emissions'].mean()/1e6:.2f} Mt CO2")
print(f"  Inertia:   {my_mc['avg_inertia'].mean():.2f} s")

## What Did We Learn?

### Price
- **Nuclear** provides the strongest price reduction per GW because it displaces gas (the marginal unit) around the clock.
- **Solar** has diminishing returns — beyond ~30%, midday prices collapse and curtailment rises.
- **Storage** flattens the price curve (reduces peaks, raises troughs) but doesn't lower the average much.

### Emissions
- **Nuclear** is the most effective decarbonization tool per GW — it displaces gas 24/7.
- **Solar+wind** reduce emissions proportionally to how much gas they displace, but their intermittency limits the effect.
- **Coal** is the worst emitter — the CO2 price is the main lever to keep it out of the merit order.

### Resilience
- In a **gas crisis**, nuclear-heavy mixes are much more resilient (smaller price increase).
- **Interconnections** provide insurance — access to French nuclear or Swiss hydro buffers against domestic price spikes.
- **Storage** is most valuable in high-renewable scenarios where price spreads are largest.

### Inertia
- **Nuclear** adds synchronous inertia (H=6s), improving grid stability.
- **Solar/wind** provide zero inertia — high penetration requires compensating mechanisms (BESS synthetic inertia, synchronous condensers).

### The real world is more complex
This simulator captures the key first-order effects, but real energy planning also involves:
- Transmission constraints (this model is copper-plate)
- Unit commitment (startup/shutdown sequencing)
- Political and social acceptance
- Construction timelines and supply chains
- Demand-side flexibility

The goal is not to prescribe a single optimal mix, but to build **quantitative intuition** about the trade-offs involved in energy transition.